In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Created on Sun Aug 25 20:05:09 2024
Modified on Thu Mar 26 2026

@author: Pablo Andrés Monroy D'Croz

================================================================================
DISCLAIMER
================================================================================
This software is provided for research purposes only. The authors and
institutions are not liable for any direct or indirect damages arising from its
use. It is not intended for clinical or diagnostic applications.

================================================================================
MIT LICENSE
================================================================================
Copyright (c) 2024-2026 Pablo Andrés Monroy D'Croz

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

================================================================================
INSTITUTIONS
================================================================================
- Universitat Pompeu Fabra (Barcelona, Spain)
- Universidad Icesi (Cali, Colombia)

================================================================================
EEGNet - ONLINE CNN/LSTM MODEL TRAINING FOR AFFECTIVE MUSIC GENERATION
================================================================================
Jupyter Notebook for training an EEGNet-inspired neural network for real-time
emotion classification from EEG signals (AF7/AF8 electrodes).

This notebook is designed to be called from the main experiment script using:
    subprocess.run(["ipython", "../eegnet_train.ipynb", participant])

The notebook:
- Receives participant code as a command-line argument
- Loads preprocessed training data from the main experiment
- Defines and trains a compact CNN+LSTM architecture
- Saves the trained model for online inference
- Exits with appropriate status codes

================================================================================
ACKNOWLEDGMENTS & ORIGINAL IMPLEMENTATION
================================================================================
This EEGNet implementation is based on the original work by Lawhern et al. (2018):
    "EEGNet: A Compact Convolutional Network for EEG-based Brain-Computer Interfaces"
    Journal of Neural Engineering, 15(5):056013.
    DOI: 10.1088/1741-2552/aace8c

The code architecture was inspired by and adapted from the official EEGNet 
implementation available at:
    https://github.com/IoBT-VISTEC/Pre-trained-EEG-for-Deep-Learning/blob/master/EEGModels.py

Original implementation features:
    - Depthwise and separable convolutions for efficient EEG processing
    - Batch normalization and ELU activations for training stability
    - Dropout regularization to prevent overfitting
    - Optimized for raw EEG signal classification

Modifications for this experiment:
    - Adapted for 2-channel input (AF7, AF8)
    - Added LSTM layer for enhanced temporal sequence learning
    - Adjusted hyperparameters for 100Hz sampling rate
    - Modified input shape handling for 10-second windows
    - Integrated with real-time music generation pipeline

================================================================================
MODEL ARCHITECTURE (EEGNet + LSTM)
================================================================================
Input Layer: (Channels=2, Timesteps=1000, 1)
    ↓
Conv2D: F1=8 filters, kernel=(1,50), padding='same'
    ↓
BatchNormalization
    ↓
DepthwiseConv2D: depth_multiplier=4, depthwise_constraint=max_norm(1.)
    ↓
BatchNormalization + ELU Activation
    ↓
AveragePooling2D: pool=(1,4), Dropout(0.5)
    ↓
SeparableConv2D: F2=16 filters, kernel=(1,16), padding='same'
    ↓
BatchNormalization + ELU Activation
    ↓
AveragePooling2D: pool=(1,8), Dropout(0.5)
    ↓
TimeDistributed(Flatten)
    ↓
LSTM: F2=16 units, return_sequences=False
    ↓
Dense: 2 units, sigmoid activation
    ↓
Dense: 1 unit, sigmoid activation (output)

Total Trainable Parameters: ~5,000-10,000 (lightweight for real-time)

================================================================================
INPUT DATA FORMAT
================================================================================
Files expected (generated by main experiment script):
    {participant}_X_train.npy: Preprocessed training sequences
        - Shape: (samples, channels, timesteps)
        - channels = 2 (AF7, AF8)
        - timesteps = 1000 (10 seconds at 100Hz)
    
    {participant}_y_train.npy: Training labels
        - Binary labels: 0 = sad, 1 = happy

Output files:
    {participant}_eegnet_model.h5: Trained Keras model for online inference

================================================================================
USAGE
================================================================================
Command line (called from main script):
    ipython ../eegnet_train.ipynb <participant_code>

Example:
    ipython ../eegnet_train.ipynb p07

The notebook will execute automatically and save the trained model.

================================================================================
"""

import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import TimeDistributed, LSTM, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import Activation, AveragePooling2D, SeparableConv2D
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
import sys
import os

# Change working directory to project location
os.chdir("/Volumes/Academico/2026-TIC phd/software/participants/")


# ============================================================================
# Model Definition
# ============================================================================

def train_eegnet_model(X_train, y_train, plot_curves=False):
    """
    Train an EEGNet-inspired neural network for emotion classification.
    
    This implementation is based on the EEGNet architecture originally proposed
    by Lawhern et al. (2018) for EEG-based Brain-Computer Interfaces. The code
    was adapted from the official implementation available at:
    https://github.com/IoBT-VISTEC/Pre-trained-EEG-for-Deep-Learning/blob/master/EEGModels.py
    
    Modifications for this experiment:
        - Added LSTM layer for enhanced temporal sequence learning
        - Adapted for 2-channel input (AF7, AF8) instead of full EEG cap
        - Adjusted kernel sizes for 100Hz sampling rate (original was 128Hz)
        - Removed channel dimension for compatibility with our data format
        - Added TimeDistributed wrapper for sequence processing
    
    Parameters
    ----------
    X_train : numpy.ndarray
        Training data with shape (samples, channels, timesteps)
        - channels: 2 (AF7, AF8)
        - timesteps: 1000 (10 seconds at 100Hz)
    y_train : numpy.ndarray
        Training labels (0=sad, 1=happy)
    plot_curves : bool, default=False
        If True, display loss and accuracy curves (disabled for batch execution)
    
    Returns
    -------
    model : tensorflow.keras.Model
        Trained Keras model ready for inference
    
    References
    ----------
    Lawhern, V. J., Solon, A. J., Waytowich, N. R., Gordon, S. M., Hung, C. P.,
    & Lance, B. J. (2018). EEGNet: A compact convolutional neural network for
    EEG-based brain-computer interfaces. Journal of Neural Engineering, 15(5), 056013.
    DOI: 10.1088/1741-2552/aace8c
    """
    
    # Extract dimensions from input data
    timesteps = X_train.shape[2]  # Number of time points
    n_channels = X_train.shape[1]  # Number of EEG channels (2: AF7, AF8)
    
    # Model hyperparameters (adapted from EEGNet)
    n_classes = 2  # Binary classification: happy vs. sad
    dropout_rate = 0.5  # Dropout rate for regularization
    kernel_length = 50  # Temporal kernel length (0.5 seconds at 100Hz)
    F1 = 8  # Number of temporal filters (original EEGNet: 4 for small dataset)
    depth_multiplier = 4  # Depthwise convolution multiplier (original EEGNet: 2)
    F2 = 16  # Number of pointwise filters (F1 * depth_multiplier)
    
    # --------------------------------------------------------------------
    # Build the EEGNet architecture (based on original implementation)
    # --------------------------------------------------------------------
    
    # Input layer: (channels, timesteps, 1)
    # Note: Original EEGNet expects (1, channels, samples) but we use (channels, samples, 1)
    input_layer = Input(shape=(n_channels, timesteps, 1))
    
    # Block 1: Temporal convolution + Depthwise convolution
    # This block learns temporal filters and spatial filters separately
    block1 = Conv2D(F1, (1, kernel_length), padding='same', 
                    use_bias=False)(input_layer)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D((n_channels, 1), use_bias=False, 
                             depth_multiplier=depth_multiplier,
                             depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)  # ELU activation for better gradient flow
    block1 = AveragePooling2D((1, 4))(block1)  # Downsample by factor of 4
    block1 = Dropout(dropout_rate)(block1)
    
    # Block 2: Separable convolution for feature combination
    # SeparableConv2D = DepthwiseConv2D + PointwiseConv2D (more efficient)
    block2 = SeparableConv2D(F2, (1, 16), use_bias=False, padding='same')(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = AveragePooling2D((1, 8))(block2)  # Further downsampling
    block2 = Dropout(dropout_rate)(block2)
    
    # Temporal flattening for LSTM (our addition to original EEGNet)
    # Allows the model to capture long-term temporal dependencies
    flatten = TimeDistributed(Flatten(name='flatten'))(block2)
    
    # LSTM layer for temporal sequence learning (custom addition)
    # This layer processes the temporal dynamics of extracted features
    lstm = LSTM(F2, return_sequences=False)(flatten)
    
    # Dense layers for final classification
    dense = Dense(n_classes, activation='sigmoid')(lstm)
    output_layer = Dense(1, activation='sigmoid')(dense)
    
    # Create and compile the model
    model = Model(input_layer, output_layer)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Train the model
    print(f"\n📊 Training EEGNet model on {X_train.shape[0]} samples...")
    print(f"   Input shape: {X_train.shape}")
    print(f"   Timesteps: {timesteps} samples ({timesteps/100:.1f} seconds at 100Hz)")
    print(f"   Channels: {n_channels} (AF7, AF8)")
    print(f"   Architecture: EEGNet + LSTM (inspired by Lawhern et al. 2018)\n")
    
    history = model.fit(X_train, y_train, epochs=10, verbose=1, batch_size=32)
    
    # Plot training curves if requested (disabled for batch execution)
    if plot_curves:
        fig, axs = plt.subplots(1, 2, figsize=(12, 4))
        
        # Loss curve
        axs[0].plot(history.history['loss'], 'b-', linewidth=2)
        axs[0].set_title('Model Loss', fontsize=14, fontweight='bold')
        axs[0].set_xlabel('Epoch')
        axs[0].set_ylabel('Loss')
        axs[0].grid(True, alpha=0.3)
        axs[0].legend(['Training'], loc='best')
        
        # Accuracy curve
        axs[1].plot(history.history['accuracy'], 'g-', linewidth=2)
        axs[1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
        axs[1].set_xlabel('Epoch')
        axs[1].set_ylabel('Accuracy')
        axs[1].grid(True, alpha=0.3)
        axs[1].legend(['Training'], loc='best')
        
        plt.tight_layout()
        plt.show()
    
    # Evaluate final model performance
    loss, accuracy = model.evaluate(X_train, y_train, verbose=0)
    print(f"\n{'='*60}")
    print(f"✅ Training Results:")
    print(f"   Timesteps: {timesteps} samples ({timesteps/100:.1f} seconds)")
    print(f"   Loss: {loss:.4f}")
    print(f"   Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"{'='*60}\n")
    
    return model


# ============================================================================
# Main Execution (Called from subprocess)
# ============================================================================

def main():
    """
    Main execution function for the Jupyter notebook.
    
    Receives participant code from command-line argument, loads training data,
    trains the EEGNet model, and saves it to disk.
    
    This function is designed to be called from the main experiment script via:
        subprocess.run(["ipython", "../eegnet_train.ipynb", participant])
    """
    
    # Get participant code from command-line arguments
    # Called as: ipython ../eegnet_train.ipynb <participant_code>
    if len(sys.argv) < 2:
        print("❌ Error: Please provide participant code as command-line argument")
        print("   Usage: ipython ../eegnet_train.ipynb <participant_code>")
        print("   Example: ipython ../eegnet_train.ipynb p07")
        sys.exit(1)
    
    participant = sys.argv[1]
    
    print(f"\n{'='*60}")
    print(f"🧠 EEGNet Training for Participant: {participant}")
    print(f"{'='*60}\n")
    print("Implementation based on Lawhern et al. (2018) EEGNet")
    print("Original code: https://github.com/IoBT-VISTEC/Pre-trained-EEG-for-Deep-Learning\n")
    
    # Load preprocessed training data
    print("📂 Loading training data...")
    try:
        X_train = np.load(participant + '_X_train.npy')
        y_train = np.load(participant + '_y_train.npy')
        print(f"   ✓ X_train shape: {X_train.shape}")
        print(f"   ✓ y_train shape: {y_train.shape}")
        print(f"   ✓ Samples: {X_train.shape[0]}")
        print(f"   ✓ Channels: {X_train.shape[1]} (AF7, AF8)")
        print(f"   ✓ Timesteps: {X_train.shape[2]} ({X_train.shape[2]/100:.1f} seconds at 100Hz)")
    except FileNotFoundError as e:
        print(f"\n❌ Error: Training data not found for participant '{participant}'")
        print(f"   Expected files: {participant}_X_train.npy and {participant}_y_train.npy")
        print("   Please run the main experiment script first to generate training data.")
        sys.exit(1)
    except Exception as e:
        print(f"\n❌ Unexpected error loading data: {e}")
        sys.exit(1)
    
    # Check if data is valid
    if X_train.shape[0] == 0:
        print("\n❌ Error: No training samples found. X_train is empty.")
        sys.exit(1)
    
    if len(np.unique(y_train)) < 2:
        print("\n❌ Error: Training data must contain both classes (sad and happy).")
        print(f"   Unique classes found: {np.unique(y_train)}")
        sys.exit(1)
    
    # Train the EEGNet model
    print("\n🏋️ Training EEGNet model...")
    try:
        modelo = train_eegnet_model(X_train, y_train, plot_curves=False)
    except Exception as e:
        print(f"\n❌ Error during model training: {e}")
        sys.exit(1)
    
    # Save the trained model
    try:
        model_filename = participant + '_eegnet_model.h5'
        modelo.save(model_filename)
        
        # Get file size
        file_size = os.path.getsize(model_filename) / 1024
        
        print(f"\n{'='*60}")
        print(f"💾 Model saved successfully!")
        print(f"   Filename: {model_filename}")
        print(f"   Size: {file_size:.2f} KB")
        print(f"{'='*60}")
        print("\nTraining complete! The model is ready for online inference.\n")
        
    except Exception as e:
        print(f"\n❌ Error saving model: {e}")
        sys.exit(1)
    
    # Optional: Display model architecture summary (commented by default)
    # print("\n📋 Model Architecture Summary:")
    # print("-" * 40)
    # modelo.summary()
    
    return 0


# ============================================================================
# Notebook Execution
# ============================================================================

if __name__ == "__main__":
    # Execute main function and exit with appropriate status code
    exit_code = main()
    sys.exit(exit_code)